In [343]:
import os
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn import metrics
vectorizer = HashingVectorizer()
from pathlib import Path
classifier = SGDClassifier(warm_start=False)
import joblib
import numpy as np

In [344]:
def save_model(classifier, vectorizer, model_path):
    joblib.dump(classifier, os.path.join(model_path, 'classifier.pkl'))
    joblib.dump(vectorizer, os.path.join(model_path, 'vectorizer.pkl'))

In [345]:
def load_model(model_path):
    classifier = joblib.load(os.path.join(model_path, 'classifier.pkl'))
    vectorizer = joblib.load(os.path.join(model_path, 'vectorizer.pkl'))
    return classifier, vectorizer

In [346]:
def train_model(training_path=None, transfer_learning=False, model_path=None, author_notes_pairs=None):
    if not training_path and not author_notes_pairs:
        raise ValueError("Either 'training_path' or 'author_notes_pairs' must be provided")
    if transfer_learning and model_path:
        old_classifier, old_vectorizer = load_model(model_path)
        vectorizer = old_vectorizer
    else:
        vectorizer = HashingVectorizer()
        classifier = SGDClassifier(warm_start=False)
    corpus = []
    documents_term_matrixes = []
    authors_per_document = []
    if training_path:
        for file in os.listdir(training_path):
            if (Path(os.path.join(training_path, file))).is_dir():
                for fileName in os.listdir(os.path.join(training_path, file)):
                    with open(os.path.join(training_path, file, fileName), 'r') as f:
                        text = f.read()
                        corpus.append(text.lower())
                        authors_per_document.append(file)
    else:
        for author, text in author_notes_pairs:
            corpus.append(text.lower())
            authors_per_document.append(author)
    documents_term_matrixes = vectorizer.fit_transform(corpus)
    if transfer_learning and model_path:
        classifier = SGDClassifier(warm_start=True)
        old_coef = old_classifier.coef_
        classes = []
        new_class_amount = 0
        for author in authors_per_document:
            if author not in old_classifier.classes_ and author not in classes:
                classes.append(author)
                new_class_amount += 1
        old_intercept = old_classifier.intercept_
        new_intercept = old_intercept
        new_coef = np.concatenate([old_coef, np.zeros((new_class_amount-old_coef.shape[0], old_coef.shape[1]))])
        new_intercept = np.concatenate([new_intercept, np.zeros(new_class_amount-old_intercept.shape[0])])
        classifier.fit(documents_term_matrixes, authors_per_document, coef_init=new_coef, intercept_init=new_intercept)
    else:
        classifier.fit(documents_term_matrixes, authors_per_document)
    predictions = []
    for document in documents_term_matrixes:
        predictions.append(classifier.predict(document)[0])
    f1 = metrics.f1_score(authors_per_document, predictions, average='macro')
    print(f"F1 Score: {f1}")
    score = classifier.score(documents_term_matrixes, authors_per_document)
    print(f"Accuracy: {score}")
    return classifier, vectorizer

In [347]:
classifier, vectorizer = train_model('./reuter+50+50/C50train/')

F1 Score: 0.9887994239541373
Accuracy: 0.9888


In [348]:
def test_model(classifier, vectorizer, testing_path):
    test_corpus = []
    test_authors = []
    test_matrix = []
    for file in os.listdir(testing_path):
        if (Path(os.path.join(testing_path, file))).is_dir():
            for fileName in os.listdir(os.path.join(testing_path, file)):
                with open(os.path.join(testing_path, file, fileName), 'r') as f:
                    text = f.read()
                    test_corpus.append(text.lower())
                    test_authors.append(file)
    test_matrix = vectorizer.transform(test_corpus)
    predictions = []
    for i, document in enumerate(test_corpus):
        X_test = test_matrix[i]
        predicted_author = classifier.predict(X_test)
        predictions.append(predicted_author[0])
    f1 = metrics.f1_score(test_authors, predictions, average='macro')
    print(f"F1 Score: {f1}")
    score = classifier.score(test_matrix, test_authors)
    print(f"Accuracy: {score}")

In [349]:
test_model(classifier, vectorizer, './reuter+50+50/C50test/')

F1 Score: 0.6796521230661732
Accuracy: 0.6828


In [350]:
save_model(classifier, vectorizer, './models/')

In [351]:
# classifier, vectorizer = train_model('./reuter+50+50/C50train/', transfer_learning=True, model_path='./models/')

In [352]:
# test_model(classifier, vectorizer, './reuter+50+50/C50test/', transfer_learning=True)

In [353]:
# classifier, vectorizer = load_model('./models/')